# TML — Gallica Year Reconstruction Universal
Multi-Colab safe: every runtime gets a unique worker ID and atomic page leases. ALTO is always active even without GPU; when CUDA is available the runtime also contributes RapidOCR / Layout compute without duplicating other Colab workers.

**Private key:** Google Drive is the default source. If the key is missing, the notebook asks for the usual manual upload once, saves that private key automatically to `MyDrive/TML/keys/tml_colab_key`, and reuses it on later sessions. The key is never stored in this repository.


In [ ]:
YEAR=1906
POLL_SECONDS=30
ALTO_DELAY=12.0
ALTO_JITTER_MAX=0.25
RAPID_WORKERS=4
RAPID_DOWNLOADERS=4
LAYOUT_WORKERS=1
LAYOUT_DOWNLOADERS=4
KEY_SOURCE='gdrive'  # 'gdrive' oppure 'upload'\nGDRIVE_KEY_PATH='/content/drive/MyDrive/TML/keys/tml_colab_key'\nALLOW_KEY_UPLOAD_FALLBACK=True\nVPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
import subprocess,uuid
from pathlib import Path
_wid=Path('/content/tml_colab_worker_id')
if _wid.exists(): WORKER_ID=_wid.read_text().strip()
else:
    WORKER_ID='colab-'+uuid.uuid4().hex[:10]
    _wid.write_text(WORKER_ID)
HAS_GPU=False; GPU_NAME='NONE'; GPU_MEM='0'
try:
    gpu_line=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True,stderr=subprocess.DEVNULL).strip().splitlines()[0]
    GPU_NAME,GPU_MEM=[x.strip() for x in gpu_line.rsplit(',',1)]; HAS_GPU=True
except Exception:
    pass
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
print('TML_SESSION_CONFIG','YEAR',YEAR,'WORKER_ID',WORKER_ID,'GPU',GPU_NAME,'GPU_AVAILABLE',HAS_GPU,'ALTO_ALWAYS_ON',True,'MULTICOLAB',True,'ALTO_DELAY',ALTO_DELAY,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
RAPID_ENV='/content/tml-rapid-env'
LAYOUT_ENV='/content/tml-layout-env'
RAPID_PY=f'{RAPID_ENV}/bin/python'
LAYOUT_PY=f'{LAYOUT_ENV}/bin/python'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','paramiko>=3.5,<4'],check=True)
if HAS_GPU:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    UV=shutil.which('uv'); assert UV
    if not os.path.exists(RAPID_PY): subprocess.run([UV,'venv','--seed',RAPID_ENV],check=True)
    subprocess.run([RAPID_PY,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
    subprocess.run([RAPID_PY,'-c',"import onnxruntime as o; p=o.get_available_providers(); print('RAPID_GPU_ENV_READY',o.__version__,p); assert 'CUDAExecutionProvider' in p"],check=True)
    subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
else:
    print('NO_GPU: compute lane disabled; ALTO downloader will still run continuously',flush=True)


In [ ]:
from pathlib import Path
import os

KEY_FILE='/content/tml_colab_key'

def _validate_private_key_bytes(data, label='private key'):
    if not data or not data.strip():
        raise RuntimeError(f'{label} is empty')
    head=data[:200]
    if b'PRIVATE KEY' not in head:
        raise RuntimeError(f'{label} does not look like a private key')

def _install_private_key(src_path):
    src=Path(src_path).expanduser()
    if not src.is_file():
        raise FileNotFoundError(f'Private key not found: {src}')
    if src.name.endswith('.pub'):
        raise RuntimeError('Configured key points to a .pub file; the private key is required')
    data=src.read_bytes()
    _validate_private_key_bytes(data, str(src))
    Path(KEY_FILE).write_bytes(data)
    os.chmod(KEY_FILE,0o600)
    print('KEY_READY',str(src),'->',KEY_FILE,'WORKER_ID',WORKER_ID,flush=True)

def _upload_once_and_persist_to_drive():
    from google.colab import files
    print('KEY_NOT_IN_DRIVE: choose the SAME private key file you used to upload manually before.',flush=True)
    uploaded=files.upload()
    if not uploaded:
        raise RuntimeError('No private key supplied')
    name,data=next(iter(uploaded.items()))
    if name.endswith('.pub'):
        raise RuntimeError('Upload the private key, not .pub')
    _validate_private_key_bytes(data, name)
    drive_path=Path(GDRIVE_KEY_PATH)
    drive_path.parent.mkdir(parents=True,exist_ok=True)
    drive_path.write_bytes(data)
    Path(KEY_FILE).write_bytes(data)
    os.chmod(KEY_FILE,0o600)
    print('KEY_SAVED_TO_DRIVE',str(drive_path),flush=True)
    print('KEY_READY_UPLOAD_ONCE',name,'WORKER_ID',WORKER_ID,flush=True)

if KEY_SOURCE.lower()=='gdrive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    try:
        _install_private_key(GDRIVE_KEY_PATH)
    except Exception as e:
        if not ALLOW_KEY_UPLOAD_FALLBACK:
            raise
        print('GDRIVE_KEY_UNAVAILABLE',type(e).__name__,str(e),flush=True)
        _upload_once_and_persist_to_drive()
elif KEY_SOURCE.lower()=='upload':
    from google.colab import files
    uploaded=files.upload()
    if not uploaded: raise RuntimeError('No private key supplied')
    name,data=next(iter(uploaded.items()))
    if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
    _validate_private_key_bytes(data, name)
    Path(KEY_FILE).write_bytes(data)
    os.chmod(KEY_FILE,0o600)
    print('KEY_READY_UPLOAD',name,'WORKER_ID',WORKER_ID,flush=True)
else:
    raise ValueError(f'Unsupported KEY_SOURCE={KEY_SOURCE!r}')


In [ ]:
import importlib.util,subprocess,sys
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip()
print('CODE',commit,'WORKER_ID',WORKER_ID,flush=True)
if HAS_GPU:
    path=f'{REPO}/colab/compute_supervisor.py'
    spec=importlib.util.spec_from_file_location('tml_compute_supervisor',path)
    mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    argv=['compute_supervisor.py','--year',str(YEAR),'--worker-id',WORKER_ID,'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--repo',REPO,'--rapid-python',RAPID_PY,'--layout-python',LAYOUT_PY,'--compute-device','cuda','--rapid-workers',str(RAPID_WORKERS),'--rapid-downloaders',str(RAPID_DOWNLOADERS),'--layout-workers',str(LAYOUT_WORKERS),'--layout-downloaders',str(LAYOUT_DOWNLOADERS),'--poll',str(POLL_SECONDS),'--alto-delay',str(ALTO_DELAY),'--alto-jitter-max',str(ALTO_JITTER_MAX)]
    print('STARTING MULTICOLAB ALTO_ALWAYS_ON + GPU_COMPUTE',WORKER_ID,flush=True)
    old_argv=sys.argv[:]; sys.argv=argv
    try: mod.main()
    finally: sys.argv=old_argv
else:
    cmd=[sys.executable,'-u',f'{REPO}/colab/alto_watch_filekey.py','--year',str(YEAR),'--worker-id',WORKER_ID,'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--delay',str(ALTO_DELAY),'--jitter-min','0','--jitter-max',str(ALTO_JITTER_MAX),'--poll',str(max(5,POLL_SECONDS))]
    print('STARTING MULTICOLAB ALTO_ALWAYS_ON ONLY (NO GPU REQUIRED)',WORKER_ID,flush=True)
    raise SystemExit(subprocess.call(cmd))
